# ☕ Mybrew Real-Time Eventstream Simulator (04)

Simulates live event data and **publishes it through the `mybrew_rt_eventstream` Eventstream**
(Custom Endpoint sources) into the existing Eventhouse KQL DB `mybrew_eventhouse` —
instead of writing to the KQL DB directly via `.ingest inline` (see `03_RT_Streaming_Simulator`
for the direct-ingest variant, which is left untouched).

**Architecture:** 4 independent pipelines inside one Eventstream item:

```
gen_pos_event()         -> pos_source (CustomEndpoint)         -\
gen_impression_event()  -> impressions_source (CustomEndpoint)  -> main_stream (DefaultStream, union)
gen_weather_event()     -> weather_source (CustomEndpoint)      -> {pos,impressions,weather,roi}_filter (event_type ==)
gen_roi_event()         -> roi_source (CustomEndpoint)          -/    -> {pos,impressions,weather,roi}_dest (Eventhouse, ProcessedIngestion)
                                                                          -> POSTransactions / AdImpressions / WeatherEvents / ROIMetrics
```

**No embedded secrets.** Publishing uses an **Azure AD (Entra ID) token** obtained via
`mssparkutils.credentials.getToken('https://eventhubs.azure.net')` — the same Fabric-native
token-acquisition pattern `03_RT_Streaming_Simulator` already uses for its Kusto calls, just
against a different audience. No SAS connection string / key is ever read, stored, or embedded:
Fabric does not expose the Custom Endpoint's SAS key via any documented public API (confirmed by
directly probing the live topology API), so this notebook deliberately avoids that credential type
entirely and relies purely on the caller's own Fabric/Entra identity, which the Eventstream's
auto-provisioned Event Hub-compatible endpoint accepts for AAD-authenticated publishers.

**Run Cell 1 (dependency install) then Cell 2 (setup) then Cell 3 (stream).** Cell 2's main loop **runs indefinitely** —
stop it manually (Interrupt/Stop the cell, or `Kernel > Interrupt`) to see graceful final stats.
Open the existing **Mybrew RTI Dashboard** to watch data arrive; no changes were made to the
dashboard, `03_RT_Streaming_Simulator`, or `mybrew_eventstream`.


In [ ]:
# Cell 1: Setup - resolves the Eventstream + KQL DB coordinates via Fabric APIs (no hardcoded secrets)
import random
import time
import json
import requests
from datetime import datetime, timezone

# ============ ENVIRONMENT PARAMETERS (externalize per dev/test/prod) ============
WORKSPACE_ID = '4bf66efa-94d7-4753-a5c4-70c79358bbd6'      # BrewP workspace
EVENTSTREAM_NAME = 'mybrew_rt_eventstream'                  # newly created Eventstream (not mybrew_eventstream)
KUSTO_URI = 'https://trd-t1s1uap7p3xkuvdx9s.z8.kusto.fabric.microsoft.com'
DB_NAME = 'mybrew_eventhouse'
FABRIC_API = 'https://api.fabric.microsoft.com/v1'

# ============ TOKEN HELPERS (Fabric built-in auth; nothing hardcoded/stored) ============
def get_token(audience):
    """Acquire a Fabric-native AAD token for the given resource audience.
    Uses mssparkutils (available automatically inside Fabric notebooks)."""
    return mssparkutils.credentials.getToken(audience)

def run_kql_query(query):
    """Execute a read-only KQL query via REST (used only for verification, not ingestion)."""
    token = get_token('https://kusto.kusto.windows.net')
    headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}
    body = {'db': DB_NAME, 'csl': query}
    r = requests.post(f'{KUSTO_URI}/v1/rest/query', headers=headers, json=body)
    if r.status_code == 200:
        return r.json()
    print(f'  KQL query error: {r.text[:200]}')
    return None

# ============ IDEMPOTENT DISCOVERY: resolve the Eventstream item + its live source endpoints ============
def resolve_eventstream_sources():
    """Look up mybrew_rt_eventstream by name (idempotent - no hardcoded item ID) and return
    {source_name: (fully_qualified_namespace, eventhub_name)} for its 4 CustomEndpoint sources.
    Uses only the documented Items-list API + the Eventstream topology API - no secrets returned
    by either call (SAS keys are masked at the API level; we never request or need them)."""
    token = get_token('https://api.fabric.microsoft.com')
    headers = {'Authorization': f'Bearer {token}'}

    items = requests.get(f'{FABRIC_API}/workspaces/{WORKSPACE_ID}/items', headers=headers).json()
    es = next((i for i in items['value'] if i['type'] == 'Eventstream' and i['displayName'] == EVENTSTREAM_NAME), None)
    if not es:
        raise RuntimeError(f'Eventstream "{EVENTSTREAM_NAME}" not found in workspace {WORKSPACE_ID}. '
                            f'Create it before running this notebook (see deployment plan).')
    es_id = es['id']

    topo = requests.get(f'{FABRIC_API}/workspaces/{WORKSPACE_ID}/eventstreams/{es_id}/topology', headers=headers).json()
    coords = {}
    for src in topo['sources']:
        conn = requests.get(
            f'{FABRIC_API}/workspaces/{WORKSPACE_ID}/eventstreams/{es_id}/sources/{src["id"]}/connection',
            headers=headers
        ).json()
        coords[src['name']] = (conn['fullyQualifiedNamespace'], conn['eventHubName'])
    return es_id, coords

EVENTSTREAM_ID, SOURCE_COORDS = resolve_eventstream_sources()
print(f'Resolved Eventstream "{EVENTSTREAM_NAME}" -> id {EVENTSTREAM_ID}')
for name, (ns, eh) in SOURCE_COORDS.items():
    print(f'  {name}: {ns} / {eh}')

# ============ PUBLISH VIA EVENT HUBS REST (no external packages, no SAS key) ============
# The Eventstream Custom Endpoint accepts plain Entra ID bearer tokens over HTTPS, so events are
# published with `requests`, which ships with the Fabric Spark runtime.
#
# Root cause of Spark_User_UserApp_ModuleNotFoundError: the previous version imported
# `azure.eventhub`, which is NOT part of the default Fabric Spark runtime. Installing it inline
# with %pip is also unreliable, because inline installs are disabled by default for runs
# triggered by the Jobs API, a pipeline activity, or the CLI. Publishing over the documented
# Event Hubs REST endpoint removes the dependency entirely, so the notebook works identically
# in interactive and scheduled/API-triggered runs.
EH_AUDIENCE = 'https://eventhubs.azure.net'
BATCH_CONTENT_TYPE = 'application/vnd.microsoft.servicebus.json'

_token_cache = {}

def _eh_headers():
    """Return auth headers with a cached Entra token (refreshed well before expiry)."""
    now = time.time()
    tok, expires_at = _token_cache.get(EH_AUDIENCE, (None, 0.0))
    if tok is None or now >= expires_at:
        tok = get_token(EH_AUDIENCE)
        _token_cache[EH_AUDIENCE] = (tok, now + 2400)
    return {'Authorization': f'Bearer {tok}', 'Content-Type': BATCH_CONTENT_TYPE}

def _endpoint(source_name):
    if source_name not in SOURCE_COORDS:
        raise RuntimeError(f'Unknown Eventstream source "{source_name}". '
                           f'Available: {sorted(SOURCE_COORDS)}')
    ns, eh = SOURCE_COORDS[source_name]
    return f'https://{ns}/{eh}/messages?api-version=2014-01'

def publish_events(source_name, event_dicts):
    """Publish a list of JSON events to one Eventstream source as a single batch."""
    if not event_dicts:
        return
    payload = [{'Body': json.dumps(ev)} for ev in event_dicts]
    r = requests.post(_endpoint(source_name), headers=_eh_headers(),
                      data=json.dumps(payload).encode('utf-8'), timeout=60)
    if r.status_code not in (200, 201):
        raise RuntimeError(f'Publish to {source_name} failed: HTTP {r.status_code} {r.text[:300]}')

def publish_event(source_name, event_dict):
    """Publish a single JSON event to one Eventstream source."""
    publish_events(source_name, [event_dict])

# ============ MYBREW DATA DIMENSIONS (identical to 03_RT_Streaming_Simulator) ============
CUSTOMERS = [
    'C-000001','C-000002','C-000003','C-000004','C-000005',
    'C-000006','C-000007','C-000008','C-000009','C-000010',
    'C-000011','C-000012','C-000015','C-000018','C-000020'
]

STORES = {
    'London': ['STR-LDN-001','STR-LDN-002','STR-LDN-003','STR-LDN-004','STR-LDN-005'],
    'Manchester': ['STR-MAN-001','STR-MAN-002'],
    'Los Angeles': ['STR-LA-001','STR-LA-002'],
    'San Francisco': ['STR-SF-001'],
    'New York': ['STR-NYC-001','STR-NYC-002'],
    'Seattle': ['STR-SEA-001']
}

PRODUCTS_HOT = [('PRD-001','Hazelnut Flat White',4.75), ('PRD-005','Chai Tea Latte',4.50),
                ('PRD-006','Toffee Nut Crunch',5.50), ('PRD-008','Matcha Oat Latte',5.25),
                ('PRD-010','Gingerbread Latte',5.50), ('PRD-011','Americano',3.50)]
PRODUCTS_COLD = [('PRD-002','Iced Caramel Macchiato',5.25), ('PRD-004','Mango Dragonfruit',4.95),
                 ('PRD-007','Cold Brew Nitro',5.00), ('PRD-009','Vanilla Sweet Cream',5.25)]

CAMPAIGNS = {
    'CMP-WW-2026': {'name':'Winter Warmth','publisher':'Meta','cpa_base':21.14},
    'CMP-WW-2026-G': {'name':'Winter Warmth','publisher':'Google','cpa_base':32.00},
    'CMP-WW-2026-A': {'name':'Winter Warmth','publisher':'Amazon','cpa_base':26.92},
    'CMP-SS-2026': {'name':'Summer Sips','publisher':'Meta','cpa_base':18.50},
    'CMP-WB-2026': {'name':'Win-Back Q3','publisher':'Direct','cpa_base':5.00},
    'CMP-TN-2026': {'name':'Tier Nudge','publisher':'Direct','cpa_base':3.50}
}

CREATIVES = {'Meta':'CR-WW-Stories-01','Google':'CR-WW-Display-01',
             'Amazon':'CR-WW-Sponsored-01','Direct':'CR-WB-Push-01'}

# Weather simulation - London is COLD (drives personalization!)
WEATHER_CITIES = [
    {'city':'London',       'temp_base':3,  'condition':'Snow/Sleet', 'precip_base':12.0},
    {'city':'London',       'temp_base':4,  'condition':'Rain',       'precip_base':8.5},
    {'city':'Manchester',   'temp_base':4,  'condition':'Rain',       'precip_base':9.8},
    {'city':'Manchester',   'temp_base':5,  'condition':'Sleet',      'precip_base':7.2},
    {'city':'Los Angeles',  'temp_base':28, 'condition':'Sunny',      'precip_base':0.0},
    {'city':'San Francisco','temp_base':18, 'condition':'Fog',        'precip_base':0.0},
    {'city':'New York',     'temp_base':20, 'condition':'Cloudy',     'precip_base':2.0},
    {'city':'Seattle',      'temp_base':17, 'condition':'Overcast',   'precip_base':0.8}
]

def _ts():
    return datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

# ============ EVENT GENERATORS (same logic as 03; emit dicts w/ event_type discriminator) ============
def gen_pos_event():
    """Generate a POS transaction event -> routed to POSTransactions via pos_filter"""
    city = random.choices(
        ['London','Manchester','Los Angeles','San Francisco','New York','Seattle'],
        weights=[50, 10, 15, 8, 12, 5]  # London-heavy for demo
    )[0]
    store = random.choice(STORES[city])

    weather_info = next((w for w in WEATHER_CITIES if w['city'] == city), WEATHER_CITIES[0])
    if weather_info['temp_base'] < 10:
        pid, pname, price = random.choice(PRODUCTS_HOT)
    else:
        pid, pname, price = random.choice(PRODUCTS_HOT + PRODUCTS_COLD)

    campaign = random.choice(list(CAMPAIGNS.keys())) if random.random() < 0.7 else ''
    amount = round(price + random.uniform(-0.20, 1.50), 2)

    return {
        'event_type': 'pos',
        'txn_id': f'TXN-{int(time.time()*1000)}{random.randint(10,99)}',
        'customer_id': random.choice(CUSTOMERS),
        'store_id': store,
        'campaign_id': campaign,
        'product_id': pid,
        'amount_gbp': amount,
        'event_time': _ts(),
    }

def gen_impression_event():
    """Generate an ad impression event -> routed to AdImpressions via impressions_filter"""
    cmp_id = random.choice(list(CAMPAIGNS.keys()))
    cmp = CAMPAIGNS[cmp_id]
    pub = cmp['publisher']

    ctr = {'Meta': 0.18, 'Google': 0.07, 'Amazon': 0.12, 'Direct': 0.40}[pub]
    clicked = random.random() < ctr
    converted = clicked and random.random() < 0.14

    spend = round(random.uniform(0.15, 2.50), 2) if pub != 'Direct' else round(random.uniform(0.01, 0.05), 2)

    return {
        'event_type': 'impression',
        'impression_id': f'IMP-{int(time.time()*1000)}{random.randint(10,99)}',
        'campaign_id': cmp_id,
        'customer_id': random.choice(CUSTOMERS),
        'publisher': pub,
        'creative_id': CREATIVES[pub],
        'clicked': clicked,
        'converted': converted,
        'spend_amount_gbp': spend,
        'event_time': _ts(),
    }

def gen_weather_event():
    """Generate a weather event -> routed to WeatherEvents via weather_filter"""
    w = random.choice(WEATHER_CITIES)
    temp = round(w['temp_base'] + random.uniform(-2.0, 2.0), 1)
    precip = round(max(0, w['precip_base'] + random.uniform(-2, 4)), 1)
    event = {
        'event_type': 'weather',
        'city': w['city'],
        'temp_c': temp,
        'condition': w['condition'],
        'precip_mm': precip,
        'event_time': _ts(),
    }
    return event, temp < 5

def gen_roi_event():
    """Generate aggregated ROI metric -> routed to ROIMetrics via roi_filter"""
    cmp_id = random.choice(list(CAMPAIGNS.keys()))
    cmp = CAMPAIGNS[cmp_id]
    pub = cmp['publisher']
    regions = ['London','Manchester','West Coast','East Coast']
    region = random.choice(regions)

    spend = round(random.uniform(5000, 50000), 2)
    roas = {'Meta': 4.2, 'Google': 2.8, 'Amazon': 3.4, 'Direct': 4.9}[pub]
    roas_var = roas + random.uniform(-0.5, 0.5)
    revenue = round(spend * roas_var, 2)
    conversions = int(spend / cmp['cpa_base'] * random.uniform(0.8, 1.2))

    return {
        'event_type': 'roi',
        'campaign_id': cmp_id,
        'publisher': pub,
        'region': region,
        'attributed_revenue_gbp': revenue,
        'total_spend_gbp': spend,
        'roi': round(roas_var, 2),
        'conversions': conversions,
        'event_time': _ts(),
    }

# ============ CONNECTIVITY CHECK (resolves endpoints + token; writes NO rows) ============
# Validates the full auth/resolution path without inserting placeholder rows into the
# dashboard tables. Real data starts flowing only when you run the streaming cell below.
print('Validating Eventstream publish path (Entra token, no SAS key, no extra packages)...')
_eh_headers()
for _required in ('pos_source', 'impressions_source', 'weather_source', 'roi_source'):
    _endpoint(_required)

print('Setup complete: all 4 sources resolved and an Entra token was acquired.')
print(f'   Eventstream: {EVENTSTREAM_NAME} ({EVENTSTREAM_ID})')
print(f'   Target Eventhouse/KQL DB: {DB_NAME}')
print('   Run the next cell to start streaming; stop it manually to end the run.')


In [ ]:
# Cell 2: START STREAMING - publishes via mybrew_rt_eventstream (runs indefinitely until interrupted)
# Interrupt this cell (Stop button / Kernel > Interrupt) to end the run - it will print final stats.

INTERVAL_SECONDS = 5    # batch every 5 seconds (same cadence as 03_RT_Streaming_Simulator)

print('\u2550' * 70)
print('  \u2615 MYBREW REAL-TIME EVENTSTREAM SIMULATOR')
print('  \u2500' * 37)
print(f'  Push interval: {INTERVAL_SECONDS}s | Runs until manually stopped (KeyboardInterrupt)')
print(f'  Publish path: mybrew_rt_eventstream (Custom Endpoint sources, AAD token)')
print(f'  Target: {DB_NAME} KQL DB (POSTransactions, AdImpressions, WeatherEvents, ROIMetrics)')
print('  ')
print('  \U0001F449 Open "Mybrew RTI Dashboard" to watch live!')
print('  \U0001F449 Stop this cell manually when done - it will print a graceful summary.')
print('\u2550' * 70)
print()

start_time = time.time()
batch_count = 0
cold_trigger_count = 0
totals = {'pos': 0, 'impressions': 0, 'weather': 0, 'roi': 0,
          'revenue': 0.0, 'ad_spend': 0.0}

def _print_final_summary(reason):
    elapsed = time.time() - start_time
    print()
    print('\u2550' * 70)
    print(f'  \u2705 SIMULATION STOPPED ({reason})')
    print(f'  Elapsed: {elapsed:.0f}s | Batches: {batch_count}')
    print('  \u2500' * 37)
    print(f'  \U0001F4B3 POS Transactions:  {totals["pos"]}')
    print(f'  \U0001F4F1 Ad Impressions:    {totals["impressions"]}')
    print(f'  \U0001F326\uFE0F Weather Events:    {totals["weather"]}')
    print(f'  \U0001F4CA ROI Updates:       {totals["roi"]}')
    print('  \u2500' * 37)
    print(f'  Total Revenue:  \u00a3{totals["revenue"]:,.2f}')
    print(f'  Total Ad Spend: \u00a3{totals["ad_spend"]:,.2f}')
    print(f'  Simulated ROAS: {totals["revenue"]/max(totals["ad_spend"],1):.2f}x')
    print(f'  \u2744\uFE0F Cold Triggers:  {cold_trigger_count} (personalization fired!)')
    print('\u2550' * 70)

try:
    while True:
        batch_count += 1
        now_str = datetime.now(timezone.utc).strftime('%H:%M:%S')

        # Generate batch of events (same volumes/cadence as 03_RT_Streaming_Simulator)
        n_pos = random.randint(4, 10)
        n_imp = random.randint(12, 25)
        n_weather = random.randint(1, 3)

        pos_events = [gen_pos_event() for _ in range(n_pos)]
        imp_events = [gen_impression_event() for _ in range(n_imp)]
        weather_results = [gen_weather_event() for _ in range(n_weather)]
        weather_events = [w[0] for w in weather_results]
        cold_triggers = sum(1 for w in weather_results if w[1])
        cold_trigger_count += cold_triggers

        # Every 3rd batch, also publish a ROI metric update
        roi_events = [gen_roi_event()] if batch_count % 3 == 0 else []

        # Publish to the Eventstream (Custom Endpoint sources) - NOT direct .ingest inline
        publish_events('pos_source', pos_events)
        publish_events('impressions_source', imp_events)
        publish_events('weather_source', weather_events)
        publish_events('roi_source', roi_events)

        # Track totals
        totals['pos'] += n_pos
        totals['impressions'] += n_imp
        totals['weather'] += n_weather
        totals['roi'] += len(roi_events)
        batch_rev = sum(e['amount_gbp'] for e in pos_events)
        batch_spend = sum(e['spend_amount_gbp'] for e in imp_events)
        totals['revenue'] += batch_rev
        totals['ad_spend'] += batch_spend

        # Print batch summary
        trigger_str = f' | \u2744\uFE0F COLD x{cold_triggers}!' if cold_triggers else ''
        roi_str = f' | \U0001F4CA ROI' if roi_events else ''
        print(f'[{now_str}] Batch {batch_count:3d} | '
              f'\U0001F4B3 {n_pos} POS (\u00a3{batch_rev:.0f}) | '
              f'\U0001F4F1 {n_imp} ads (\u00a3{batch_spend:.1f}) | '
              f'\U0001F326\uFE0F {n_weather} wx'
              f'{trigger_str}{roi_str}')

        time.sleep(INTERVAL_SECONDS)

except KeyboardInterrupt:
    _print_final_summary('KeyboardInterrupt / manually stopped')
except Exception as e:
    print(f'\n\u274c Unexpected error: {type(e).__name__}: {e}')
    _print_final_summary('error')
    raise


In [ ]:
# Cell 3: Verify results - query KQL tables (same verification pattern as 03_RT_Streaming_Simulator)
print('\n\U0001F4CA KQL Table Counts After Streaming:')
print('-' * 40)
tables = ['POSTransactions', 'AdImpressions', 'WeatherEvents', 'ROIMetrics']
for t in tables:
    result = run_kql_query(f'{t} | count')
    if result:
        count = result['Tables'][0]['Rows'][0][0]
        print(f'  {t}: {count} rows')

print('\n\n\U0001F4C8 Live ROI by Publisher:')
print('-' * 50)
result = run_kql_query("""
AdImpressions
| summarize Imp=count(), Clicks=countif(clicked), Conv=countif(converted),
            Spend=round(sum(spend_amount_gbp),2) by publisher
| extend CTR_pct=round(100.0*todouble(Clicks)/todouble(Imp), 1),
         CPA=round(Spend/todouble(max_of(Conv,1)), 2)
| order by Imp desc
""")
if result:
    print(f'{"Publisher":<10}{"Impressions":<13}{"Clicks":<8}{"CTR%":<7}{"Conv":<7}{"Spend":<12}{"CPA"}')
    for row in result['Tables'][0]['Rows']:
        print(f'{row[0]:<10}{row[1]:<13}{row[2]:<8}{row[4]:<7}{row[3]:<7}\u00a3{row[5]:<10}\u00a3{row[6]}')

print('\n\n\u2744\uFE0F Cold Weather Triggers (Personalization):')
print('-' * 50)
result = run_kql_query("""
WeatherEvents
| where temp_c < 5
| summarize ColdEvents=count(), AvgTemp=round(avg(temp_c),1) by city
| order by ColdEvents desc
""")
if result:
    for row in result['Tables'][0]['Rows']:
        print(f'  {row[0]}: {row[1]} cold events (avg {row[2]}\u00b0C) \u2192 Hazelnut Flat White pushed!')
